In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors gradio

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from PIL import Image
import os
import time
import gc

from typing import Optional, Tuple

from datetime import datetime

from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionXLPipeline,
    EulerAncestralDiscreteScheduler,
    EulerDiscreteScheduler,
    DPMSolverMultistepScheduler,
    DDIMScheduler,
    LMSDiscreteScheduler
)

import gradio as gr

In [ ]:
class StableDiffusionGenerator:

    def __init__(
        self,
        model_id: str = "runwayml/stable-diffusion-v1-5",
        device: str = "auto"
    ):

        try:

            self.device = self._setup_device(device)

            self.dtype = (
                torch.float16
                if self.device.type == "cuda"
                else torch.float32
            )

            print(f"Initializing Stable Diffusion on {self.device}")
            print(f"Using precision: {self.dtype}")

            self.pipe = self._load_pipeline(model_id)

            self.current_scheduler = "euler_a"

            self.schedulers = {
                "euler_a": (
                    "Euler Ancestral",
                    "Fast, good for creative images"
                ),

                "euler": (
                    "Euler",
                    "Deterministic, consistent results"
                ),

                "ddim": (
                    "DDIM",
                    "Classic, good quality, slower"
                ),

                "dpm_solver": (
                    "DPM Solver",
                    "High quality, efficient"
                ),

                "lms": (
                    "LMS",
                    "Linear multistep, stable"
                )
            }

            print("Stable Diffusion Generator Ready!")

        except Exception as e:

            print(f"Initialization Error: {str(e)}")

            raise


    # Device Setup
    def _setup_device(self, device: str) -> torch.device:

        if device == "auto":

            if torch.cuda.is_available():

                device = "cuda"

                print(
                    f"GPU Detected: "
                    f"{torch.cuda.get_device_name(0)}"
                )

                vram_gb = (
                    torch.cuda.get_device_properties(0).total_memory
                    / 1024**3
                )

                print(f"VRAM: {vram_gb:.1f} GB")

            else:

                device = "cpu"

                print("Using CPU (GPU not available)")

        return torch.device(device)


    # Pipeline Loading
    def _load_pipeline(self, model_id: str):

        try:

            # Select the correct pipeline

            if "RealVisXL" in model_id or "XL" in model_id:

                print("Detected SDXL model.")

                pipe = StableDiffusionXLPipeline.from_pretrained(
                    model_id,
                    torch_dtype=self.dtype,
                    use_safetensors=True
                )

            else:

                print("Detected Stable Diffusion model.")

                pipe = StableDiffusionPipeline.from_pretrained(
                    model_id,
                    torch_dtype=self.dtype,
                    safety_checker=None,
                    use_safetensors=True
                )


            print("Model loaded successfully.")
            print("Applying memory optimizations...")


            # Attention Slicing

            try:

                pipe.enable_attention_slicing()

                print("Attention Slicing: Enabled")

            except Exception as e:

                print(
                    f"Attention Slicing not available: {e}"
                )


            # VAE Slicing
            #
            # Some Diffusers versions expose this directly
            # through the pipeline.
            #
            # Other versions expose it through pipe.vae.

            try:

                if hasattr(pipe, "enable_vae_slicing"):

                    pipe.enable_vae_slicing()

                    print("VAE Slicing: Enabled")

                elif hasattr(pipe.vae, "enable_slicing"):

                    pipe.vae.enable_slicing()

                    print("VAE Slicing: Enabled through VAE")

                else:

                    print("VAE Slicing: Not available")

            except Exception as e:

                print(
                    f"VAE Slicing not available: {e}"
                )


            # XFormers

            try:

                pipe.enable_xformers_memory_efficient_attention()

                print("XFormers Attention: Enabled")

            except Exception as e:

                print(
                    f"XFormers: Not available ({e})"
                )


            # GPU


            if self.device.type == "cuda":

                try:

                    pipe = pipe.to(self.device)

                    print("Full GPU Loading: Success")

                except RuntimeError as e:

                    print(
                        f"GPU Memory Limited: {e}"
                    )

                    print(
                        "Using CPU Offloading..."
                    )

                    try:

                        pipe.enable_model_cpu_offload()

                        print(
                            "CPU Offloading: Enabled"
                        )

                    except Exception as offload_error:

                        raise RuntimeError(
                            "GPU loading failed and CPU "
                            f"offloading also failed: {offload_error}"
                        )

            # CPU

            else:

                pipe = pipe.to(self.device)

                print("Running on CPU")


            return pipe


        except Exception as e:

            raise RuntimeError(
                f"Failed to load model: {e}"
            )


    # Scheduler

    def set_scheduler(
        self,
        scheduler_name: str
    ) -> bool:

        if scheduler_name not in self.schedulers:

            print(
                f"Unknown scheduler: {scheduler_name}"
            )

            return False


        if scheduler_name == self.current_scheduler:

            return True


        scheduler_map = {

            "euler_a":
                EulerAncestralDiscreteScheduler,

            "euler":
                EulerDiscreteScheduler,

            "ddim":
                DDIMScheduler,

            "dpm_solver":
                DPMSolverMultistepScheduler,

            "lms":
                LMSDiscreteScheduler

        }


        try:

            scheduler_class = scheduler_map[
                scheduler_name
            ]

            self.pipe.scheduler = (
                scheduler_class.from_config(
                    self.pipe.scheduler.config
                )
            )

            self.current_scheduler = scheduler_name

            name, desc = self.schedulers[
                scheduler_name
            ]

            print(
                f"Scheduler Changed: "
                f"{name} ({desc})"
            )

            return True


        except Exception as e:

            print(
                f"Scheduler Error: {e}"
            )

            return False



    # Image Generation

    def generate_image(
        self,
        prompt: str,
        negative_prompt: str = "",
        width: int = 512,
        height: int = 512,
        num_inference_steps: int = 20,
        guidance_scale: float = 7.5,
        seed: Optional[int] = None,
        scheduler: str = "euler_a"
    ) -> Tuple[Image.Image, dict]:

        # Validate prompt

        if not prompt.strip():

            raise ValueError(
                "Prompt cannot be empty"
            )


        # Set scheduler

        self.set_scheduler(scheduler)


        # Generate seed


        if seed is None:

            seed = torch.randint(
                0,
                2**32,
                (1,)
            ).item()


        # Random Generator

        generator = torch.Generator(
            device=self.device
        )

        generator.manual_seed(seed)


        # Stable Diffusion dimensions
        #
        # Width and height must be divisible by 8.

        width = (width // 8) * 8
        height = (height // 8) * 8


        print(
            f"Generating: '{prompt[:50]}...'"
        )

        print(
            f"Size: {width}x{height}, "
            f"Steps: {num_inference_steps}, "
            f"CFG: {guidance_scale}"
        )

        print(
            f"Seed: {seed}, "
            f"Scheduler: {scheduler}"
        )


        start_time = time.time()


        try:

            # Inference


            with torch.inference_mode():

                if (
                    self.device.type == "cuda"
                    and self.dtype == torch.float16
                ):

                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16
                    ):

                        result = self.pipe(

                            prompt=prompt,

                            negative_prompt=(
                                negative_prompt
                                if negative_prompt
                                else None
                            ),

                            width=width,

                            height=height,

                            num_inference_steps=(
                                num_inference_steps
                            ),

                            guidance_scale=(
                                guidance_scale
                            ),

                            generator=generator
                        )

                else:

                    result = self.pipe(

                        prompt=prompt,

                        negative_prompt=(
                            negative_prompt
                            if negative_prompt
                            else None
                        ),

                        width=width,

                        height=height,

                        num_inference_steps=(
                            num_inference_steps
                        ),

                        guidance_scale=(
                            guidance_scale
                        ),

                        generator=generator
                    )


            # Generation Time

            generation_time = (
                time.time() - start_time
            )


            # Metadata

            metadata = {

                "prompt": prompt,

                "negative_prompt":
                    negative_prompt,

                "width":
                    width,

                "height":
                    height,

                "steps":
                    num_inference_steps,

                "guidance_scale":
                    guidance_scale,

                "scheduler":
                    scheduler,

                "seed":
                    seed,

                "generation_time":
                    round(generation_time, 2),

                "device":
                    str(self.device),

                "dtype":
                    str(self.dtype)

            }


            print(
                f"Generated in "
                f"{generation_time:.2f}s"
            )


            return (
                result.images[0],
                metadata
            )


        except torch.cuda.OutOfMemoryError:

            self._cleanup_memory()

            raise RuntimeError(

                "GPU Out of Memory! "
                "Try reducing image size, "
                "using fewer steps, "
                "or using CPU mode."
            )


        except Exception as e:

            raise RuntimeError(
                f"Generation failed: {str(e)}"
            )


        finally:

            self._cleanup_memory()


    # Memory Cleanup


    def _cleanup_memory(self):

        gc.collect()

        if self.device.type == "cuda":

            torch.cuda.empty_cache()


    # Memory Usage

    def get_memory_usage(self) -> dict:

        memory_info = {}


        if self.device.type == "cuda":

            memory_info = {

                "allocated_gb":
                    torch.cuda.memory_allocated()
                    / 1024**3,

                "reserved_gb":
                    torch.cuda.memory_reserved()
                    / 1024**3,

                "max_allocated_gb":
                    torch.cuda.max_memory_allocated()
                    / 1024**3,

                "total_gb":
                    torch.cuda.get_device_properties(0)
                    .total_memory
                    / 1024**3

            }

        else:

            memory_info = {

                "device":
                    "cpu",

                "note":
                    "CPU memory tracking not available"

            }


        return memory_info


    # Save Image

    def save_image(
        self,
        image: Image.Image,
        metadata: dict,
        output_dir: str = "outputs"
    ) -> str:

        os.makedirs(
            output_dir,
            exist_ok=True
        )


        timestamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )


        filename = (

            f"sd_gen_"
            f"{timestamp}_"
            f"s{metadata['seed']}_"
            f"{metadata['width']}x"
            f"{metadata['height']}.png"

        )


        filepath = os.path.join(
            output_dir,
            filename
        )


        image.save(filepath)



        # Save metadata

        metadata_file = filepath.replace(
            ".png",
            "_metadata.txt"
        )


        with open(
            metadata_file,
            "w"
        ) as f:

            f.write(
                "Stable Diffusion Generation Metadata\n"
            )

            f.write(
                "=" * 40 + "\n"
            )

            for key, value in metadata.items():

                f.write(
                    f"{key}: {value}\n"
                )


        print(
            f"Saved: {filepath}"
        )


        return filepath

In [ ]:
class StableDiffusionUI:

    def __init__(self):

        self.generator = None

        self.gallery_images = []

        self.generation_history = []


    # Initialize Generator

    def initialize_generator(
        self,
        model_choice: str,
        device_choice: str
    ) -> str:

        try:

            model_map = {

                "Stable Diffusion 1.5 (Recommended)":
                    "runwayml/stable-diffusion-v1-5",

                "Stable Diffusion 2.1":
                    "stabilityai/stable-diffusion-2-1",

                "Realistic Vision (RealVisXL)":
                    "SG161222/RealVisXL_V4.0"

            }


            device_map = {

                "Auto (Recommended)":
                    "auto",

                "GPU (CUDA)":
                    "cuda",

                "CPU (Slower)":
                    "cpu"

            }


            model_id = model_map.get(

                model_choice,

                "runwayml/stable-diffusion-v1-5"

            )


            device = device_map.get(

                device_choice,

                "auto"

            )


            self.generator = StableDiffusionGenerator(

                model_id=model_id,

                device=device

            )


            memory_info = (
                self.generator.get_memory_usage()
            )


            memory_text = (

                f"Memory Usage: "
                f"{memory_info}"

                if memory_info

                else

                "Ready!"

            )


            return (
                "Model loaded successfully!\n"
                f"{memory_text}"
            )


        except Exception as e:

            return (
                f"Initialization failed: {str(e)}"
            )


    # Image Generation Handler

    def generate_image(

        self,

        prompt: str,

        negative_prompt: str,

        width: int,

        height: int,

        steps: int,

        guidance: float,

        scheduler: str,

        seed: int,

        save_image: bool

    ) -> Tuple[Optional[Image.Image], str, str]:


        if self.generator is None:

            return (
                None,
                "Please initialize the model first!",
                ""
            )


        if not prompt.strip():

            return (
                None,
                "Please enter a prompt!",
                ""
            )


        try:

            # Seed -1 means random seed

            seed = (
                None
                if seed == -1
                else int(seed)
            )


            image, metadata = (
                self.generator.generate_image(

                    prompt=prompt,

                    negative_prompt=negative_prompt,

                    width=width,

                    height=height,

                    num_inference_steps=steps,

                    guidance_scale=guidance,

                    scheduler=scheduler,

                    seed=seed

                )
            )


            # Format information

            info_text = (
                self._format_generation_info(
                    metadata
                )
            )


            saved_path = ""


            # Save image

            if save_image:

                saved_path = (
                    self.generator.save_image(
                        image,
                        metadata
                    )
                )



            # History

            self.generation_history.append(
                metadata
            )

            self.gallery_images.append(
                image
            )


            # Keep only latest 10 images

            if len(self.gallery_images) > 10:

                self.gallery_images = (
                    self.gallery_images[-10:]
                )

                self.generation_history = (
                    self.generation_history[-10:]
                )


            return (
                image,
                info_text,
                saved_path
            )


        except Exception as e:

            return (
                None,
                f"Generation failed: {str(e)}",
                ""
            )


    # Format Generation Information

    def _format_generation_info(
        self,
        metadata: dict
    ) -> str:

        return f"""
Generation Complete!

Parameters Used:

- Prompt: {metadata['prompt'][:100]}{'...' if len(metadata['prompt']) > 100 else ''}
- Size: {metadata['width']} x {metadata['height']} pixels
- Steps: {metadata['steps']} (more steps = higher quality, slower)
- Guidance Scale: {metadata['guidance_scale']} (higher = follows prompt more closely)
- Scheduler: {metadata['scheduler']}
- Seed: {metadata['seed']} (for reproducible results)

Performance:

- Generation Time: {metadata['generation_time']}s
- Device: {metadata['device']}
- Precision: {metadata['dtype']}
"""



    # Example Prompts

    def get_example_prompts(self) -> list:

        return [

            [
                "a serene mountain landscape at sunrise, "
                "photorealistic, highly detailed",

                "blurry, low quality"
            ],

            [
                "portrait of a wise old wizard, "
                "fantasy art, digital painting",

                "ugly, deformed"
            ],

            [
                "cyberpunk cityscape at night, "
                "neon lights, futuristic",

                "daytime, bright"
            ],

            [
                "cute cartoon cat wearing a hat, "
                "kawaii style",

                "realistic, scary"
            ],

            [
                "abstract geometric patterns, "
                "colorful, modern art",

                "representational, dull colors"
            ]

        ]


    # Scheduler Information

    def show_scheduler_info(
        self,
        scheduler: str
    ) -> str:

        scheduler_info = {

            "euler_a":
                "Euler Ancestral: Fast and creative, "
                "adds slight randomness for variety",

            "euler":
                "Euler: Deterministic and consistent, "
                "same seed = same result",

            "ddim":
                "DDIM: Classic scheduler, "
                "high quality but slower",

            "dpm_solver":
                "DPM Solver: Efficient high-quality generation",

            "lms":
                "LMS: Linear multistep, "
                "very stable results"

        }


        return scheduler_info.get(

            scheduler,

            "Scheduler information not available"

        )


    # Memory Information

    def get_memory_info(self) -> str:

        if self.generator is None:

            return "Model not loaded"


        try:

            memory_info = (
                self.generator.get_memory_usage()
            )


            if "allocated_gb" in memory_info:

                return f"""
GPU Memory Usage:

- Allocated: {memory_info['allocated_gb']:.2f} GB
- Reserved: {memory_info['reserved_gb']:.2f} GB
- Total Available: {memory_info['total_gb']:.2f} GB
- Usage: {(memory_info['allocated_gb'] / memory_info['total_gb'] * 100):.1f}%
"""


            else:

                return (
                    "CPU mode - memory tracking "
                    "not available"
                )


        except Exception:

            return "Memory info unavailable"


    # Create Gradio Interface

    def create_interface(self) -> gr.Blocks:

        with gr.Blocks(

            title="Educational Stable Diffusion Generator",

            theme=gr.themes.Soft()

        ) as interface:



            # Header

            gr.Markdown(
                """
# Educational Stable Diffusion Text-to-Image Generator

**Learn Generative AI concepts while creating images!**
"""
            )


            # Setup & Generation Tab


            with gr.Tab("Setup & Generation"):


                # Model Setup / System Info

                with gr.Row():


                    with gr.Column():

                        gr.Markdown(
                            "### Model Setup"
                        )


                        model_choice = gr.Dropdown(

                            choices=[

                                "Stable Diffusion 1.5 (Recommended)",

                                "Stable Diffusion 2.1",

                                "Realistic Vision (RealVisXL)"

                            ],

                            value=(
                                "Stable Diffusion 1.5 "
                                "(Recommended)"
                            ),

                            label="Model Selection"

                        )


                        device_choice = gr.Dropdown(

                            choices=[

                                "Auto (Recommended)",

                                "GPU (CUDA)",

                                "CPU (Slower)"

                            ],

                            value="Auto (Recommended)",

                            label="Device Selection"

                        )


                        init_btn = gr.Button(

                            "Initialize Model",

                            variant="primary"

                        )


                        init_status = gr.Textbox(

                            label="Initialization Status",

                            placeholder=(
                                "Click Initialize Model "
                                "to start"
                            ),

                            lines=3

                        )


                    with gr.Column():


                        gr.Markdown(
                            "### System Info"
                        )


                        memory_btn = gr.Button(
                            "Check Memory Usage"
                        )


                        memory_info = gr.Textbox(

                            label="Memory Information",

                            placeholder=(
                                "Click to check memory usage"
                            ),

                            lines=6

                        )


                # Image Generation

                gr.Markdown(
                    "### Image Generation"
                )


                with gr.Row():


                    with gr.Column():


                        prompt = gr.Textbox(

                            label=(
                                "Prompt "
                                "(Describe what you want)"
                            ),

                            placeholder=(
                                "a beautiful landscape "
                                "painting, oil on canvas, detailed"
                            ),

                            lines=3

                        )


                        negative_prompt = gr.Textbox(

                            label=(
                                "Negative Prompt "
                                "(What to avoid)"
                            ),

                            placeholder=(
                                "blurry, low quality, "
                                "bad anatomy"
                            ),

                            lines=2

                        )


                        generate_btn = gr.Button(

                            "Generate Image",

                            variant="primary",

                            size="lg"

                        )


                    with gr.Column():


                        with gr.Accordion(

                            "Advanced Settings",

                            open=True

                        ):


                            with gr.Row():

                                width = gr.Slider(

                                    256,
                                    1024,
                                    512,
                                    step=64,
                                    label="Width"

                                )


                                height = gr.Slider(

                                    256,
                                    1024,
                                    512,
                                    step=64,
                                    label="Height"

                                )


                            with gr.Row():

                                steps = gr.Slider(

                                    10,
                                    100,
                                    20,
                                    step=1,
                                    label="Inference Steps"

                                )


                                guidance = gr.Slider(

                                    1.0,
                                    20.0,
                                    7.5,
                                    step=0.5,
                                    label="Guidance Scale"

                                )


                            scheduler = gr.Dropdown(

                                choices=[

                                    "euler_a",

                                    "euler",

                                    "ddim",

                                    "dpm_solver",

                                    "lms"

                                ],

                                value="euler_a",

                                label="Scheduler"

                            )


                            scheduler_info = gr.Textbox(

                                label="Scheduler Information",

                                interactive=False,

                                lines=2

                            )


                            with gr.Row():


                                seed = gr.Number(

                                    -1,

                                    label="Seed"

                                )


                                save_image = gr.Checkbox(

                                    True,

                                    label=(
                                        "Save Generated Images"
                                    )

                                )



                # Output

                with gr.Row():

                    output_image = gr.Image(

                        label="Generated Image",

                        type="pil"

                    )


                with gr.Row():


                    generation_info = gr.Textbox(

                        label="Generation Information",

                        lines=10,

                        interactive=False

                    )


                    saved_path = gr.Textbox(

                        label="Saved File Path",

                        interactive=False

                    )


            # Learning Resources Tab

            with gr.Tab("Learning Resources"):


                gr.Markdown(
                    """
## Understanding Stable Diffusion

### What is Diffusion?

Diffusion models learn to gradually remove noise from random data.

### Key Components:

**CLIP (Text Encoder)**

**U-Net (Denoising Network)**

**VAE (Variational Autoencoder)**

**Schedulers**

### Parameter Guide:

**Steps (10-100):**
More steps = higher quality but slower generation.

**Guidance Scale (1-20):**
Higher values make the AI follow your prompt more strictly.

**Seed:**
Controls randomness. Same seed + same settings = reproducible results.

**Resolution:**
Higher resolution = more detail but requires more GPU memory.
"""
                )



            # Examples & Gallery Tab

            with gr.Tab("Examples & Gallery"):


                gr.Markdown(
                    "### Example Prompts to Try"
                )


                examples = gr.Examples(

                    examples=self.get_example_prompts(),

                    inputs=[
                        prompt,
                        negative_prompt
                    ],

                    label="Click any example to load it"

                )


                gr.Markdown(
                    "### Recent Generations"
                )


                gallery = gr.Gallery(

                    value=[],

                    label="Your Generated Images",

                    show_label=True,

                    elem_id="gallery",

                    columns=3,

                    rows=2,

                    object_fit="contain",

                    height="auto"

                )


            # Event Handlers

            # Initialize Model

            init_btn.click(

                fn=self.initialize_generator,

                inputs=[
                    model_choice,
                    device_choice
                ],

                outputs=init_status

            )


            # Generate Image

            generate_btn.click(

                fn=self.generate_image,

                inputs=[

                    prompt,

                    negative_prompt,

                    width,

                    height,

                    steps,

                    guidance,

                    scheduler,

                    seed,

                    save_image

                ],

                outputs=[

                    output_image,

                    generation_info,

                    saved_path

                ]

            ).then(

                fn=lambda: self.gallery_images,

                outputs=gallery

            )


            # Scheduler Information

            scheduler.change(

                fn=self.show_scheduler_info,

                inputs=scheduler,

                outputs=scheduler_info

            )


            # Memory Information

            memory_btn.click(

                fn=self.get_memory_info,

                outputs=memory_info

            )


        return interface


# Launch Application

ui = StableDiffusionUI()

interface = ui.create_interface()

interface.launch(

    share=True,

    server_name="0.0.0.0",

    server_port=7860,

    debug=True,

    show_error=True

)



In [3]:
import torch
import diffusers

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

PyTorch: 2.11.0+cu128
Diffusers: 0.40.0
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
